<a href="https://colab.research.google.com/github/EdgyL/ISYS2001_Assignment/blob/main/ISYS2001_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ISYS2001_Assignment

In [44]:
# Install gradio
!pip install gradio pandas

In [45]:
# Import libaries
import gradio as gr
import pandas as pd

## Load data
The following is the pseudocode for implementation of loading data:

```
load_csv(file_path)
    CHECK file exists

    TRY reading CSV into Dataframe
        IF read fails
            DISPLAY error message to user
    
    CHECK CSV contains [date, transaction_type, amount] (desc is optional)
    CONVERT dates into datetime values
    CONVERT amounts into numbers
    CHECK amounts not negative
    CHECK transaction types are recognised
    CHECK there is exactly one opening balance

    SORT transactions by date
    CHECK first transaction is the opening balance

    RETURN the Dataframe
```

In [46]:
# Function to load CSV data into a Dataframe for Gradio to use
def load_csv(file_path):
    if file_path is None:
        raise gr.Error("Please upload a CSV file.")

    try:
        df = pd.read_csv(file_path)
    except (OSError, UnicodeDecodeError, pd.errors.ParserError,
            pd.errors.EmptyDataError):
        raise gr.Error("Could not read the file. Please upload a valid CSV.")

    required_columns = ["date", "transaction_type", "amount"]

    for column in required_columns:
        if column not in df.columns:
            raise gr.Error(f"Missing required column: {column}")

    return df

## Caculate Current Balance
The following is the pseudocode logic for the `current_balance()` function:

```
FUNCTION calculate_balance(df)
    SET balance to 0

    FOR each transaction
        GET transaction type and amount

        IF transaction type is withdrawal
            SUBTRACT amount from balance
        ELSE IF transaction type is opening_balance, deposit or interest
            ADD amount to balance

    RETURN balance
```


In [47]:
def calculate_balance(df):
    balance = 0
    transactions = df.to_dict(orient="records")

    for transaction in transactions:
        transaction_type = transaction["transaction_type"]
        amount = transaction["amount"]

        if transaction_type == "withdrawal":
            balance -= amount
        elif transaction_type in ["opening_balance", "deposit", "interest"]:
            balance += amount

    return balance

## Calculate Average Weekly Contributions
The pseudocode for this:
```
FUNCTION calculate_average_weekly(df)
    GET dates from date column

    SET earliest_date to the first date
    SET latest_date to the first date

    FOR each date
        IF date is before earliest_date
            SET earliest_date to date

        IF date is after latest_date
            SET latest_date to date


    CALCULATE number_of_days as latest_date minus earliest_date
    DIVIDE number_of_days by 7 to get number_of_weeks

    IF number of weeks is 0
        RETURN 0

    SET total_deposits to 0

    FOR each transaction
        IF transaction type is deposit
            ADD amount to total_deposits

    DIVIDE total_deposits by number_of_weeks

    RETURN average weekly contribution
```

In [48]:
def calculate_average_weekly(df):
    transactions = df.to_dict(orient="records")
    dates = pd.to_datetime(df["date"])

    earliest_date = dates.iloc[0]
    latest_date = dates.iloc[0]

    # Find the earliest and latest dates
    for date in dates:
        if date < earliest_date:
            earliest_date = date

        if date > latest_date:
            latest_date = date

    num_days = (latest_date - earliest_date).days
    num_weeks = num_days / 7

    if num_weeks == 0:
        return 0

    total_deposits = 0

    for transaction in transactions:
        transaction_type = transaction["transaction_type"]
        amount = transaction["amount"]

        if transaction_type == "deposit":
            total_deposits += amount

    weekly_contributions = total_deposits / num_weeks

    return weekly_contributions


## Calculate Balance History

```
FUNCTION calculate_balance_history(df)
    CONVERT date column into dates
    SORT transactions from earliest to latest

    SET balance to 0
    CREATE empty history list

    FOR each transaction
        IF transaction type is withdrawal
            SUBTRACT amount from balance
        ELSE IF transaction type is opening_balance, deposit or interest
            ADD amount to balance

        ADD transaction date and updated balance to history

    CONVERT history into a DataFrame
    RETURN history DataFrame
```

In [49]:
def calculate_balance_history(df):
    # Make a copy so the original data stays unchanged
    history_data = df.copy()

    history_data["date"] = pd.to_datetime(history_data["date"])
    history_data = history_data.sort_values("date", kind="stable")

    transactions = history_data.to_dict(orient="records")

    balance = 0
    history = []

    for transaction in transactions:
        transaction_type = transaction["transaction_type"]
        amount = transaction["amount"]

        if transaction_type == "withdrawal":
            balance -= amount
        elif transaction_type in ["opening_balance", "deposit", "interest"]:
            balance += amount

        history.append({
            "date": transaction["date"],
            "balance": balance,
        })

    return pd.DataFrame(history)

## Calculate percentage of income contributed to savings

```
FUNCTION calculate_contribution_percentage(weekly_average, weekly_income)
    IF weekly_average or weekly_income is missing
        RETURN no value

    IF weekly_income is less than or equal to 0
        RETURN no value

    DIVIDE weekly_average by weekly_income
    MULTIPLY result by 100

    RETURN contribution percentage
```

In [50]:
def calculate_contribution_percentage(weekly_average, weekly_income):
    if weekly_average is None or weekly_income is None:
        return None

    if weekly_income <= 0:
        return None

    contribution_percentage = (weekly_average / weekly_income) * 100

    return contribution_percentage

## Projection of future savings

```
FUNCTION project_savings(starting_balance, weekly_amount, annual_rate, years)
    SET balance to starting_balance
    CALCULATE weekly_rate as annual_rate / 100 / 52
    CALCULATE total_weeks as years * 52

    CREATE history list containing week 0 and starting_balance

    FOR each week from 1 to total_weeks
        CALCULATE interest as balance * weekly_rate
        ADD interest to balance
        ADD weekly_amount to balance

        ADD week and updated balance to history

    CALCULATE total_contributions as weekly_amount * total_weeks
    CALCULATE interest_earned as balance - starting_balance - total_contributions

    CONVERT history into a DataFrame

    RETURN balance, total_contributions, interest_earned and history
```

In [51]:
def project_savings(starting_balance, weekly_amount, annual_rate, years):
    balance = starting_balance
    weekly_rate = annual_rate / 100 / 52
    total_weeks = int(years * 52)

    history = [{
        "year": 0,
        "balance": balance,
    }]

    for week in range(1, total_weeks + 1):
        interest = balance * weekly_rate
        balance += interest
        balance += weekly_amount

        history.append({
            "year": week / 52,
            "balance": balance,
        })

    total_contributions = weekly_amount * total_weeks
    interest_earned = balance - starting_balance - total_contributions

    return (
        f"${balance:,.2f}",
        f"${total_contributions:,.2f}",
        f"${interest_earned:,.2f}",
        pd.DataFrame(history),
    )


## Reset projection values
```
FUNCTION reset_projection_defaults(current_balance, weekly_average)
    IF current balance is null
        SET current balance to 0

    IF weekly average is null
        SET weekly average to 0

    return current balance, weekly average, 5% annual rate, 5 years projection length
```

In [52]:
def reset_projection_defaults(current_balance, weekly_average):
    if current_balance is None:
        current_balance = 0

    if weekly_average is None:
        weekly_average = 0

    return current_balance, weekly_average, 5, 5

## Helper Method for processing the CSV

Pseudocode:
```
FUNCTION process_csv(file_path)
    CALL load_csv and STORE DataFrame
    CALCULATE current balance
    CALCULATE average weekly contribution
    CALCULATE balance history

    RETURN balance, weekly average and history
```

In [53]:
def process_csv(file_path):
    df = load_csv(file_path)
    balance = calculate_balance(df)
    weekly_avg = calculate_average_weekly(df)
    history = calculate_balance_history(df)

    return balance, weekly_avg, history

## Test Cases

Now that the main functions have been implemented, I need to test that they calculate the correct values. For this I will use a small set of transactions that I can calculate by hand, then compare these results with the output of each function using asserts.

### Sample data

The sample transactions will be:

```text
2026-01-01: Opening balance of $1,000
2026-01-08: Deposit of $200
2026-01-15: Deposit of $200
2026-01-22: Withdrawal of $100
2026-01-29: Interest earned of $5
```

### Calculate current balance

Therefore the expected current balance of the account should be:

```text
Current balance = opening balance + deposits + interest - withdrawals
Current balance = 1000 + 200 + 200 + 5 - 100
Current balance = $1,305
```

I will call `calculate_balance()` with the sample data and use an assert to check that it returns `1305`.

### Calculate average weekly contributions

The sample data covers 28 days, which is 4 weeks. During this period the user deposited $400.

```text
Number of weeks = 28 / 7 = 4
Total deposits = 200 + 200 = $400

Average weekly contribution = 400 / 4 = $100
```

This should only include deposits, not the opening balance or interest. It should also include the weeks where the user did not deposit anything.

Therefore `calculate_average_weekly()` should return `100`.

### Calculate balance history

For the history graph, I need to check that the balance after each transaction is correct.

The expected values are:

```text
Opening balance:       $1,000
After first deposit:   $1,200
After second deposit:  $1,400
After withdrawal:      $1,300
After interest:        $1,305
```

I will compare the balance column returned by `calculate_balance_history()` with these values.

I will also test transactions entered in a different date order to check that the function sorts them correctly. Another test will use multiple transactions on the same date to check that each transaction is kept in the history.

### Calculate percentage of income contributed

For this function I will use a weekly contribution of `$200` and a weekly income of $1,000.

```text
Contribution percentage = (200 / 1000) * 100
Contribution percentage = 20%
```

Therefore the function should return `20`.

I will also test:
- A contribution of `$0` which should return `0`.
- An income of $0 or a negative income, which should return `None`.
- A missing income or weekly average, which should return `None`.

This checks that the function does not try to divide by zero or calculate with missing values.

### Reset projection defaults

The reset function should return the calculated historical values along with the default interest rate and projection length.

```text
Current balance = $1,305
Average weekly contribution = $100

Expected returned values = 1305, 100, 5, 5
```

If no CSV has been loaded, the current balance and weekly average will be `None`. In this case the expected returned values are:

```
0, 0, 5, 5
```

### Project future savings

To test the projection function I'll start with an example that has no interest since its easier to calculate.

```
Starting balance = $1,000
Weekly contribution = $200
Annual interest rate = 0%
Projection length = 1 year

Total contributions = 200 * 52 = $10,400
Projected balance = 1000 + 10400 = $11,400
Interest earned = $0
```

Since I changed the function to return monetary formatted strings I'll have to ensure the asserts need to check for `"$11,400.00"`, `"$10,400.00"` and `"$0.00"`, and not just the numbers themselves

The history should also contain 53 weeks of data as it includes the starting balance at week 0 and then 52 weekly contributions.

To check that compound interest works, I will use the following example:

```
Starting balance = $1,000
Weekly contribution = $0
Annual interest rate = 5.2%

Weekly rate = 5.2 / 100 / 52 = 0.001

Week 1 balance = 1000 + (1000 * 0.001)
Week 1 balance = $1,001

Week 2 balance = 1001 + (1001 * 0.001)
Week 2 balance = $1,002.001
```

This tests that the interest earned in the first week was added when calculating the interest for the second week. I will also test an example with a weekly contribution to check that the deposit is added after the interest.

### Loading CSV files

For load_csv() I will test the CSV uploads manually through the Gradio interface. I will upload a valid CSV and check that the calculated results match the expected values. The description column is optional, so the file should still load without it.

I will also upload:

A completely empty file.

A CSV missing one of the required columns, repeating this for date, transaction_type and amount.

Each of these should display the relevant error message. I'll put screenshots of these tests in my dev diary.

For test cases that cannot be easily tested via the upload interface on Gradio, I will call `load_csv()` directly with `None` and a file path that does not exist. Then use try-except blocks and asserts to check that the expected errors are raised.

My current checks only cover reading the file and checking its columns, so they do not confirm that every transaction contains valid data. Therefore I'm assuming the user provides a valid transaction history prepared in the application's required CSV format.

In [54]:
# 1. Create sample data
test_data = pd.DataFrame({
    "date": [
        "2026-01-01",
        "2026-01-08",
        "2026-01-15",
        "2026-01-22",
        "2026-01-29",
    ],
    "transaction_type": [
        "opening_balance",
        "deposit",
        "deposit",
        "withdrawal",
        "interest",
    ],
    "amount": [1000, 200, 200, 100, 5],
})

assert calculate_balance(test_data) == 1305

print("Passed: calculate_balance()")

# Test average weekly contributions
assert calculate_average_weekly(test_data) == 100
print("Passed: calculate_average_weekly()")

# Test balance history
history = calculate_balance_history(test_data)

assert history["balance"].tolist() == [
    1000, 1200, 1400, 1300, 1305
]
print("Passed: calculate_balance_history()")

# Test transactions supplied out of date order
unsorted_data = test_data.iloc[[2, 0, 4, 1, 3]]
history = calculate_balance_history(unsorted_data)

assert history["balance"].tolist() == [
    1000, 1200, 1400, 1300, 1305
]
print("Passed: transactions sorted correctly")

# Test multiple transactions on the same date
same_date_data = pd.DataFrame({
    "date": ["2026-01-01", "2026-01-01", "2026-01-01"],
    "transaction_type": ["opening_balance", "deposit", "withdrawal"],
    "amount": [1000, 200, 50],
})

history = calculate_balance_history(same_date_data)

assert history["balance"].tolist() == [1000, 1200, 1150]
print("Passed: transactions sharing a date are preserved")

Passed: calculate_balance()
Passed: calculate_average_weekly()
Passed: calculate_balance_history()
Passed: transactions sorted correctly
Passed: transactions sharing a date are preserved


In [55]:
# 2. Test income percentages and reset defaults
assert calculate_contribution_percentage(200, 1000) == 20
assert calculate_contribution_percentage(0, 1000) == 0

# Missing or invalid income
assert calculate_contribution_percentage(200, 0) is None
assert calculate_contribution_percentage(200, -1000) is None
assert calculate_contribution_percentage(200, None) is None
assert calculate_contribution_percentage(None, 1000) is None

# Defaults with historical values
assert reset_projection_defaults(1305, 100) == (1305, 100, 5, 5)

# Defaults without historical values
assert reset_projection_defaults(None, None) == (0, 0, 5, 5)
assert reset_projection_defaults(1305, None) == (1305, 0, 5, 5)

print("Passed: calculate_contribution_percentage() and reset_projection_defaults()")

Passed: calculate_contribution_percentage() and reset_projection_defaults()


In [56]:
# 3. Test future projections
balance, contributions, interest, history = project_savings(
    1000, 200, 0, 1
)

assert balance == "$11,400.00"
assert contributions == "$10,400.00"
assert interest == "$0.00"

assert len(history) == 53
assert history["balance"].iloc[0] == 1000
assert history["balance"].iloc[-1] == 11400
assert history["year"].iloc[-1] == 1

# No starting balance, contributions or interest
balance, contributions, interest, history = project_savings(
    0, 0, 0, 1
)

assert balance == "$0.00"
assert contributions == "$0.00"
assert interest == "$0.00"

# Weekly compounding: 5.2% annually gives 0.1% per week
balance, contributions, interest, history = project_savings(
    1000, 0, 5.2, 1
)

assert round(history["balance"].iloc[1], 3) == 1001.000
assert round(history["balance"].iloc[2], 3) == 1002.001
assert contributions == "$0.00"

# Check deposits are added AFTER interest is calculated
balance, contributions, interest, history = project_savings(
    1000, 200, 5.2, 1
)

assert round(history["balance"].iloc[1], 3) == 1201.000
assert round(history["balance"].iloc[2], 3) == 1402.201

print("Passed: project_savings()")

Passed: project_savings()


In [57]:
# 4. Test invalid file path
try:
    load_csv(None)
except gr.Error as error:
    assert "Please upload a CSV file." in str(error)
    print("Passed: missing file input raises the expected error.")
else:
    assert False, "Expected an error when no file was provided."

# Test a file path that does not exist
try:
    load_csv("file_that_does_not_exist.csv")
except gr.Error as error:
    assert "Could not read the file." in str(error)
    print("Passed: nonexistent file raises the expected error.")
else:
    assert False, "Expected an error for a nonexistent file."

assert calculate_contribution_percentage(200, None) is None
assert calculate_contribution_percentage(None, 1000) is None
assert reset_projection_defaults(None, None) == (0, 0, 5, 5)

Passed: missing file input raises the expected error.
Passed: nonexistent file raises the expected error.


## Savings assistant chatbot

For this feature I will add a chatbot that lets the user ask questions about their savings. The chatbot will have access to the current dashboard variables and be able to use these to explain the results.

I will use my existing functions to calculate the figures before sending them to the chatbot, ensuring the projection only uses the users current inputs.

The chatbot will also be able to recall prior messages so the user can ask follow up questions.

### Setting up Gemini

I will store my API key in Colab Secrets instead of writing it directly into the notebook.

To set this up:
1. Get your API key from google studio
2. Open the Secrets section in Colab using the key icon
3. Add a secret called `GEMINI_API_KEY`
4. Turn on notebook access for this secret

In [58]:
!pip install requests

import requests
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")

MODEL = "gemini-3.5-flash-lite"

API_URL = (
    "https://generativelanguage.googleapis.com/v1beta/models/"
    f"{MODEL}:generateContent"
)

SYSTEM_INSTRUCTION = """
You are a friendly savings assistant.

Explain the user's savings dashboard using simple language.
Use the supplied calculated figures when explaining results.
Treat missing values as unknown, not zero.

Keep historical account figures separate from future projections.
The latest dashboard summary replaces figures from earlier messages.

Projections assume 52 weeks per year, a constant nominal annual
interest rate, weekly compounding and contributions added at
the end of each week. They are estimates, not guaranteed returns.

Do not claim to change dashboard inputs or access bank accounts.
If a user wants a different projection, explain which inputs
they should change.
"""

In [59]:
def create_dashboard_summary(
    current_balance,
    weekly_average,
    weekly_income,
    starting_balance,
    weekly_amount,
    annual_rate,
    years,
):
    for value in [starting_balance, weekly_amount, annual_rate, years]:
        if value is None:
            raise gr.Error("Please complete all projection inputs.")
        if value < 0:
            raise gr.Error("Projection inputs cannot be negative.")

    balance, contributions, interest, _ = project_savings(
        starting_balance, weekly_amount, annual_rate, years
    )

    percentage = calculate_contribution_percentage(
        weekly_average, weekly_income
    )

    if percentage is None:
        percentage = "Not available"
    else:
        percentage = f"{percentage:.2f}%"

    return f"""
        Historical account figures:
        Current balance ($): {current_balance if current_balance is not None else "Not provided"}
        Average weekly deposits ($): {weekly_average if weekly_average is not None else "Not provided"}
        Entered weekly income ($): {weekly_income if weekly_income is not None else "Not provided"}
        Deposits as a percentage of income: {percentage}

        Projection settings:
        Starting balance ($): {starting_balance}
        Weekly contribution ($): {weekly_amount}
        Annual interest rate (%): {annual_rate}
        Years: {years}

        Projection results:
        Projected balance: {balance}
        Total future contributions: {contributions}
        Interest earned: {interest}

        These results were recalculated for this question.
        Click Update projection to make the graph match.
    """

### Sending a question to Gemini

The chatbot function will prepare the dashboard summary and previous messages, then send them to Gemini using requests.post().

Gemini uses the role "model" for chatbot responses, so I will convert Gradio's "assistant" role when preparing the conversation.

If the request fails or no response text is returned, the function will display a helpful error.

In [60]:
def chat_with_gemini(
    message,
    history,
    current_balance,
    weekly_average,
    weekly_income,
    starting_balance,
    weekly_amount,
    annual_rate,
    years,
):
    if not message or not message.strip():
        raise gr.Error("Please enter a question.")

    summary = create_dashboard_summary(
        current_balance,
        weekly_average,
        weekly_income,
        starting_balance,
        weekly_amount,
        annual_rate,
        years,
    )

    contents = []

    # Convert previous Gradio messages into Gemini's format
    for previous_message in history:
        role = previous_message["role"]

        if role == "assistant":
            role = "model"

        content = previous_message["content"]

        # Newer Gradio versions store text inside content blocks
        if isinstance(content, list):
            text_parts = []

            for block in content:
                if block.get("type") == "text":
                    text_parts.append(block["text"])

            content = "\n".join(text_parts)

        if content and role in ["user", "model"]:
            contents.append({
                "role": role,
                "parts": [{"text": content}],
            })

    # Add the latest question and current dashboard figures
    contents.append({
        "role": "user",
        "parts": [{
            "text": (
                f"Current dashboard summary:\n{summary}\n\n"
                f"User question:\n{message}"
            )
        }],
    })

    payload = {
        "systemInstruction": {
            "parts": [{"text": SYSTEM_INSTRUCTION}]
        },
        "contents": contents,
    }

    try:
        response = requests.post(
            API_URL,
            headers={"x-goog-api-key": API_KEY},
            json=payload,
            timeout=60,
        )
    except requests.RequestException:
        raise gr.Error(
            "Could not connect to Gemini. Please try again."
        )

    if response.status_code != 200:
        raise gr.Error(
            f"Gemini request failed ({response.status_code}). "
            "Check your API key, model access and usage quota."
        )

    try:
        data = response.json()
    except ValueError:
        raise gr.Error("Gemini returned an unreadable response.")

    candidates = data.get("candidates", [])

    if not candidates:
        raise gr.Error(
            "Gemini did not return a response. Try rephrasing your question."
        )

    parts = candidates[0].get("content", {}).get("parts", [])
    reply_parts = []

    for part in parts:
        if "text" in part and not part.get("thought", False):
            reply_parts.append(part["text"])

    reply = "\n".join(reply_parts)

    if not reply.strip():
        raise gr.Error("Gemini did not return any response text.")

    return reply

### Displaying the chatbot on the dashboard

I will add a `ChatInterface` below the existing Gradio dashboard. The additional inputs will supply the current dashboard values whenever the user sends a message.

The user's question, conversation history and dashboard summary will be sent to Gemini (not the original CSV)

### Testing the chatbot

I will test the chatbot through the dashboard and record screenshots of the results.

First, I will upload my sample CSV and enter a weekly income of $1,000. When I ask what percentage of my income I contribute, the chatbot should explain that `$100` per week is `10%`.

Next, I will enter these projection settings:
- Starting balance: `$1,000`
- Weekly contribution: `$200`
- Annual interest rate: `0%`
- Projection length: 1 year

The chatbot should report a projected balance of `$11,400`, total contributions of `$10,400` and interest earned of `$0`.

I will then change the annual interest rate to 5% and compare its answer with the calculated dashboard results.

I'll also ask a question that will test if it can recall conversation history.

All of this will be put into my dev diary.

## Display on Gradio Dashboard
The following is the pseudocode logic for displaying the data:
```
CREATE Gradio app
    ADD CSV upload
    ADD current balance display

    WHEN a CSV is uploaded
        LOAD CSV into a DataFrame
        CALCULATE balance using the DataFrame
        DISPLAY current balance

    ADD weekly income input
    ADD contribution percentage display

    WHEN weekly income changes
        CALCULATE percentage using average weekly contribution and income
        DISPLAY contribution percentage

    WHEN average weekly contribution changes
        CALCULATE percentage using average weekly contribution and income
        DISPLAY contribution percentage
      
    WHEN Return to defaults is clicked
        SET starting balance to calculated current balance
        SET weekly contribution to calculated weekly average
        SET annual interest rate to 5
        SET projection length to 5
    
    DISPLAY gemini chatbot
        
RUN app
```

In [62]:

with gr.Blocks() as app:
    gr.Markdown("# Savings Dashboard")

    csv_upload = gr.File(
        label="Upload savings CSV",
        file_types=[".csv"],
        type="filepath",
    )

    with gr.Row():
        current_balance = gr.Number(
            label="Current balance ($)",
            precision=2,
            interactive=False,
        )

        weekly_avg = gr.Number(
            label="Average weekly contribution ($)",
            precision=2,
            interactive=False,
        )

    with gr.Row():
        weekly_income = gr.Number(
            label="Weekly take-home income ($)",
            minimum=0,
            precision=2,
            interactive=True,
        )

        contribution_percentage = gr.Number(
            label="Income contributed to savings (%)",
            precision=2,
            interactive=False,
        )

    history_graph = gr.LinePlot(
        x="date",
        y="balance",
        title="Savings account history",
        x_title="Date",
        y_title="Balance ($)",
    )


    gr.Markdown("## Future savings projection")

    with gr.Row():
        starting_balance = gr.Number(
            label="Starting balance ($)",
            value=0,
            minimum=0,
            precision=2,
            interactive=True,
        )

        weekly_amount = gr.Number(
            label="Weekly contribution ($)",
            value=0,
            minimum=0,
            precision=2,
            interactive=True,
        )

    with gr.Row():
        annual_rate = gr.Number(
            label="Annual interest rate (%)",
            value=5,
            minimum=0,
            interactive=True,
        )

        years = gr.Slider(
            label="Projection length (years)",
            minimum=1,
            maximum=50,
            value=5,
            step=1,
        )

    with gr.Row():
        update_button = gr.Button("Update projection")
        reset_button = gr.Button("Return to defaults")

    with gr.Row():
        projected_balance = gr.Textbox(
            label="Projected balance",
            interactive=False,
        )

        total_contributions = gr.Textbox(
            label="Total future contributions",
            interactive=False,
        )

        interest_earned = gr.Textbox(
            label="Future interest earned",
            interactive=False,
        )

    projection_graph = gr.LinePlot(
        x="year",
        y="balance",
        title="Projected savings growth",
        x_title="Years",
        y_title="Balance ($)",
    )

    gr.Markdown("## Savings assistant")

    gr.Markdown(
        "Ask a question about your savings or projection. "
        "Your messages and dashboard summary will be sent to Google Gemini."
    )

    gr.ChatInterface(
        fn=chat_with_gemini,
        additional_inputs=[
            current_balance,
            weekly_avg,
            weekly_income,
            starting_balance,
            weekly_amount,
            annual_rate,
            years,
        ],
    )

    csv_upload.upload(
        fn=process_csv,
        inputs=csv_upload,
        outputs=[current_balance, weekly_avg, history_graph],
    ).success(
        fn=reset_projection_defaults,
        inputs=[current_balance, weekly_avg],
        outputs=[starting_balance, weekly_amount, annual_rate, years],
    )

    # Recalculate when the user changes their income
    weekly_income.change(
        fn=calculate_contribution_percentage,
        inputs=[weekly_avg, weekly_income],
        outputs=contribution_percentage,
    )

    # Recalculate when a CSV updates the weekly average
    weekly_avg.change(
        fn=calculate_contribution_percentage,
        inputs=[weekly_avg, weekly_income],
        outputs=contribution_percentage,
    )

    update_button.click(
        fn=project_savings,
        inputs=[starting_balance, weekly_amount, annual_rate, years],
        outputs=[
            projected_balance,
            total_contributions,
            interest_earned,
            projection_graph,
        ],
    )

    reset_button.click(
        fn=reset_projection_defaults,
        inputs=[current_balance, weekly_avg],
        outputs=[starting_balance, weekly_amount, annual_rate, years],
    )

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://8bd976e38cc773154e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
